In [1]:
import numpy as np
import pandas as pd

In [2]:
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

Step 1 — Domain understanding

In [3]:
FILE = "article_001.xlsx"          # <-- point this at your file
xl = pd.ExcelFile(FILE)
xl.sheet_names

['DATA', 'SEPTEMBER', 'SEPTEMBER BUDGET', 'SEP', 'WORKINGS']

In [4]:
workings = pd.read_excel(xl, "WORKINGS", header=None)
mix = workings.iloc[1:5, [0, 1]]
mix.columns = ["department", "weight"]
mix

,department,weight
1,Bakery,0.277
2,Butchery,0.2094
3,Deli,0.22
4,Veges,0.2935


In [6]:
print(
    "Domain summary:\n"
    "- Supermarket chain: 72 branches, each running up to 4\n"
    "  departments (Bakery, Butchery, Deli, Veges).\n"
    "- A monthly sales TARGET is set per branch-department, compared\n"
    "  day by day against ACTUAL sales.\n"
    "- Targets are built top-down: a store-level number is split across\n"
    "  departments using the fixed weights above.")

Domain summary:
- Supermarket chain: 72 branches, each running up to 4
  departments (Bakery, Butchery, Deli, Veges).
- A monthly sales TARGET is set per branch-department, compared
  day by day against ACTUAL sales.
- Targets are built top-down: a store-level number is split across
  departments using the fixed weights above.


Step 2 — Data understanding

In [7]:
raw = pd.read_excel(xl, "SEPTEMBER", header=None)
print("raw shape:", raw.shape)
raw.iloc[:2, :8]   # two-row merged header — real header is row index 1


raw shape: (292, 66)


,0,1,2,3,4,5,6,7
0,1,NaN,2,NaN,tue,wed,thu,fri
1,BRANCH,NaN,Department,NaN,4,5,6,7


In [8]:
actual = pd.read_excel(xl, "SEPTEMBER", header=1)
budget = pd.read_excel(xl, "SEPTEMBER BUDGET", header=1)
print("actual:", actual.shape, "| budget:", budget.shape)
actual.head()

actual: (290, 66) | budget: (297, 33)


,BRANCH,Unnamed: 1,Department,Unnamed: 3,4,5,6,7,8,9,...,21.1,22.1,23.1,24.1,25.1,26.1,27.1,28.1,29.1,30.1
0,KIAMBU,1.0,BAKERY,KIAMBUBAKERY,106968.15,115197.77,110605.33,110503.26,130789.39,141636.25,...,107509.121356,101092.075982,99967.440247,108319.363048,115147.226579,129251.005060,139098.041733,130324.078221,134803.674732,99469.336615
1,RUAKA,2.0,BAKERY,RUAKABAKERY,108668.15,114263.49,116284.29,117639.98,153730.91,152686.01,...,122540.792548,121950.731978,108898.383180,106084.710640,138564.908809,161448.760677,163625.239240,180125.137802,178152.145996,100265.382059
2,RUAI,3.0,BAKERY,RUAIBAKERY,143774.63,138485.43,140682.26,155139.98,195117.05,178430.01,...,135166.113051,113632.325420,105104.028362,106293.588668,124263.337760,135769.545297,156017.146175,155747.289763,149218.686515,117402.210265
3,EBP 1,4.0,BAKERY,EBP 1BAKERY,43522.58,41741.99,39663.20,53241.13,56469.71,58868.47,...,41030.908304,43624.278713,38688.066876,36904.793950,41270.557552,50869.541846,49079.107577,60357.407271,64652.993640,44799.467002
4,Kahawa,5.0,BAKERY,KahawaBAKERY,72370.54,74751.16,105980.78,83997.71,92772.89,91085.54,...,61116.742518,64355.677010,61459.099417,70862.038532,74208.674994,74219.113031,84945.495233,84240.530393,77521.532932,67196.948186


In [9]:
print("Missing (actual):", int(actual.isna().sum().sum()))
print("Missing (budget):", int(budget.isna().sum().sum()))
print("Branches:", actual["BRANCH"].nunique())
print("Departments:", sorted(actual["Department"].dropna().unique()))


Missing (actual): 532
Missing (budget): 353
Branches: 72
Departments: ['BAKERY', 'BUTCHERY', 'DELI', 'VEGES']


In [10]:
actual[["Total Actual Sales", "SEPT SALES TARGET"]].describe().round(0)

,Total Actual Sales,SEPT SALES TARGET
count,282.0,282.0
mean,4695285.0,4022347.0
std,39324501.0,33690086.0
min,55297.0,10622.0
25%,1333144.0,1132879.0
50%,2057541.0,1724321.0
75%,2806384.0,2423500.0
max,662035138.0,567150949.0


In [11]:
no_actuals = actual[actual["BRANCH"].notna() & actual["Total Actual Sales"].isna()]
print(f"Branch-departments with no actual sales: {len(no_actuals)}")
no_actuals[["BRANCH", "Department"]]


Branch-departments with no actual sales: 7


,BRANCH,Department
80,PIPELINE,BUTCHERY
82,OUTERING,BUTCHERY
126,SHABAAB,BUTCHERY
198,SHABAAB,DELI
224,PIPELINE,VEGES
243,KERICHO,VEGES
270,SHABAAB,VEGES


In [12]:
neg_values = (actual.select_dtypes("number") < 0).sum().sum()
print("Negative values found:", neg_values)

Negative values found: 0


Step 3 — Data preparation (clean)

In [13]:
actual_clean = actual.dropna(subset=["BRANCH"]).copy()
budget_clean = budget.dropna(subset=["BRANCH"]).copy()
print(f"actual: {len(actual)} -> {len(actual_clean)}")
print(f"budget: {len(budget)} -> {len(budget_clean)}")

actual: 290 -> 288
budget: 297 -> 288


In [14]:
summary = actual_clean[["BRANCH", "Department",
                        "Total Actual Sales", "SEPT SALES TARGET"]].copy()
summary.columns = ["BRANCH", "DEPARTMENT", "ACTUAL_SALES", "TARGET_SALES"]
summary["VARIANCE"] = summary["ACTUAL_SALES"] - summary["TARGET_SALES"]
summary["ACHIEVEMENT_PCT"] = (summary["ACTUAL_SALES"] / summary["TARGET_SALES"] * 100).round(1)
summary.head()

,BRANCH,DEPARTMENT,ACTUAL_SALES,TARGET_SALES,VARIANCE,ACHIEVEMENT_PCT
0,KIAMBU,BAKERY,3638290.64,3.477932e+06,160359.110731,104.6
1,RUAKA,BAKERY,3652816.35,4.052005e+06,-399188.420593,90.1
2,RUAI,BAKERY,4536250.43,3.878267e+06,657983.207676,117.0
3,EBP 1,BAKERY,1405125.26,1.382223e+06,22902.075368,101.7
4,Kahawa,BAKERY,2442092.36,2.180552e+06,261540.568920,112.0


 Step 4 — Filter

In [15]:
active_summary = summary.dropna(subset=["ACTUAL_SALES", "TARGET_SALES"])
print(f"{len(summary)} -> {len(active_summary)} rows after dropping inactive depts")


288 -> 281 rows after dropping inactive depts


In [16]:
missed_target = active_summary[active_summary["ACHIEVEMENT_PCT"] < 100]
print(f"Below target: {len(missed_target)} of {len(active_summary)}")
missed_target.head()

Below target: 46 of 281


,BRANCH,DEPARTMENT,ACTUAL_SALES,TARGET_SALES,VARIANCE,ACHIEVEMENT_PCT
1,RUAKA,BAKERY,3652816.35,4.052005e+06,-3.991884e+05,90.1
17,WAIYAKI WAY,BAKERY,1338507.10,1.633366e+06,-2.948585e+05,81.9
29,KILIMANI,BAKERY,2567562.04,2.619215e+06,-5.165297e+04,98.0
62,BANANA RD,BAKERY,1889409.15,2.786542e+06,-8.971325e+05,67.8
63,NAKURU STATEHOUSE,BAKERY,914544.24,2.727825e+06,-1.813281e+06,33.5


In [17]:
big_beat = active_summary[active_summary["ACHIEVEMENT_PCT"] > 200]
print(f"Over 200% of target: {len(big_beat)}")


Over 200% of target: 10


In [18]:
suspect = active_summary[active_summary["TARGET_SALES"] < 100_000]
print(f"Targets under KES 100,000/month (suspected errors): {len(suspect)}")
suspect[["BRANCH", "DEPARTMENT", "ACTUAL_SALES", "TARGET_SALES"]]

Targets under KES 100,000/month (suspected errors): 6


,BRANCH,DEPARTMENT,ACTUAL_SALES,TARGET_SALES
86,KONDELE,BUTCHERY,298783.10000,10622.048899
88,MAYFAIR,BUTCHERY,55296.68000,45768.580715
98,TOM MBOYA,BUTCHERY,126380.41000,37249.406944
99,KERICHO,BUTCHERY,64155.21000,31056.558370
152,PIPELINE,DELI,69229.28895,34202.124018
256,OTC,VEGES,160751.59000,28847.260986


In [19]:
worst_first = missed_target.sort_values("ACHIEVEMENT_PCT").reset_index(drop=True)
worst_first.head(10)


,BRANCH,DEPARTMENT,ACTUAL_SALES,TARGET_SALES,VARIANCE,ACHIEVEMENT_PCT
0,NAKURU KENYATTA AVENUE,VEGES,1.007228e+06,3.937514e+06,-2.930286e+06,25.6
1,NAKURU KENYATTA AVENUE,BUTCHERY,5.652147e+05,1.803072e+06,-1.237858e+06,31.3
2,NAKURU STATEHOUSE,BUTCHERY,5.785214e+05,1.803072e+06,-1.224551e+06,32.1
3,NAKURU STATEHOUSE,VEGES,1.279420e+06,3.937514e+06,-2.658094e+06,32.5
4,NAKURU STATEHOUSE,BAKERY,9.145442e+05,2.727825e+06,-1.813281e+06,33.5
5,NAKURU STATEHOUSE,DELI,1.043735e+06,2.341973e+06,-1.298239e+06,44.6
6,MAKONGENI THIKA,BUTCHERY,1.149293e+06,2.223519e+06,-1.074226e+06,51.7
7,LIKONI FERRY,VEGES,1.599272e+06,2.837171e+06,-1.237900e+06,56.4
8,ELGON VIEW,BUTCHERY,1.368150e+06,2.394513e+06,-1.026363e+06,57.1
9,NAKURU SEC 58,BUTCHERY,1.457279e+06,2.394513e+06,-9.372342e+05,60.9


In [20]:
biggest_first = active_summary.sort_values("ACTUAL_SALES", ascending=False).reset_index(drop=True)
biggest_first.head(10)


,BRANCH,DEPARTMENT,ACTUAL_SALES,TARGET_SALES,VARIANCE,ACHIEVEMENT_PCT
0,LAVINGTON,VEGES,12841233.11,1.044104e+07,2.400189e+06,123.0
1,KILELESHWA,VEGES,12207482.70,1.200065e+07,2.068302e+05,101.7
2,KILIMANI,VEGES,10119051.31,8.850051e+06,1.269001e+06,114.3
3,KILELESHWA,BUTCHERY,9952688.16,8.337466e+06,1.615222e+06,119.4
4,LAVINGTON,BUTCHERY,9499617.20,7.809296e+06,1.690321e+06,121.6
5,KIAMBU,VEGES,9172014.79,9.094053e+06,7.796144e+04,100.9
6,KIAMBU,BUTCHERY,8571812.10,7.312577e+06,1.259235e+06,117.2
7,OTC,BAKERY,7513874.70,5.806577e+06,1.707297e+06,129.4
8,MFANGANO,BAKERY,7421996.47,6.437781e+06,9.842152e+05,115.3
9,KILIMANI,BUTCHERY,7033190.39,5.741115e+06,1.292076e+06,122.5


In [21]:
by_dept = active_summary.sort_values(
    ["DEPARTMENT", "ACHIEVEMENT_PCT"], ascending=[True, False]
).reset_index(drop=True)
by_dept.groupby("DEPARTMENT").head(1) 

,BRANCH,DEPARTMENT,ACTUAL_SALES,TARGET_SALES,VARIANCE,ACHIEVEMENT_PCT
0,LIKONI FERRY,BAKERY,2.830306e+06,1.426287e+06,1.404018e+06,198.4
72,KONDELE,BUTCHERY,2.987831e+05,1.062205e+04,2.881611e+05,2812.9
141,NYALI,DELI,2.392840e+06,1.096363e+06,1.296477e+06,218.3
212,OTC,VEGES,1.607516e+05,2.884726e+04,1.319043e+05,557.3


Final regards

In [23]:
chain_actual = active_summary["ACTUAL_SALES"].sum()
chain_target = active_summary["TARGET_SALES"].sum()
chain_pct = round(chain_actual / chain_target * 100, 1)


In [24]:
print(f"Chain-wide actual sales : KES {chain_actual:,.0f}")
print(f"Chain-wide target sales : KES {chain_target:,.0f}")
print(f"Chain-wide achievement  : {chain_pct}%")


Chain-wide actual sales : KES 662,035,138
Chain-wide target sales : KES 567,150,949
Chain-wide achievement  : 116.7%


In [25]:
print(f"""Final regards:
1. Data is clean - no negative values, no duplicate branch-department rows,
   and every daily figure reconciles to its monthly total.
2. {len(no_actuals)} branch-departments have no data because they don't run
   that department. This is normal, not missing data, and was correctly
   excluded from every average above.
3. {len(suspect)} targets are under KES 100,000/month - almost certainly
   typos, since actual sales there are 2-30x the target.
4. {len(missed_target)} of {len(active_summary)} branch-departments are
   currently below target - sorted above from worst to least-worst.
5. The heaviest-trading branch-departments (sorted above) carry a large
   share of chain revenue - protect those first if you can only focus
   on a few.
""")

Final regards:
1. Data is clean - no negative values, no duplicate branch-department rows,
   and every daily figure reconciles to its monthly total.
2. 7 branch-departments have no data because they don't run
   that department. This is normal, not missing data, and was correctly
   excluded from every average above.
3. 6 targets are under KES 100,000/month - almost certainly
   typos, since actual sales there are 2-30x the target.
4. 46 of 281 branch-departments are
   currently below target - sorted above from worst to least-worst.
5. The heaviest-trading branch-departments (sorted above) carry a large
   share of chain revenue - protect those first if you can only focus
   on a few.

